In [2]:
# =========================================================
# 13B_classification_comparison.ipynb
# Compare classification models using FINAL feature set
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")
import joblib
models_dir = project_root / "Models"
models_dir.mkdir(exist_ok=True)

# -------------------------------
# Configuration
# -------------------------------
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_final_model_ready.csv",
    "TCS": data_dir / "tcs_final_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_final_model_ready.csv"
}

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True)
}

# -------------------------------
# Helper
# -------------------------------
def evaluate_classification(y_true, y_pred, y_proba=None):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }
    if y_proba is not None:
        try:
            metrics["ROC_AUC"] = roc_auc_score(y_true, y_proba)
        except:
            metrics["ROC_AUC"] = np.nan
    else:
        metrics["ROC_AUC"] = np.nan
    return metrics

# -------------------------------
# Main Loop
# -------------------------------
results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")
    if not path.exists():
        print(f"  ⚠️ Missing file: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded dataset shape: {df.shape}")

    # --- Detect correct classification target ---
    target_col = None
    for candidate in ["Target_Cls_y", "Target_Cls"]:
        if candidate in df.columns:
            target_col = candidate
            break

    if not target_col:
        print(f"  ⚠️ Skipping {ticker} — no classification target found.")
        continue
    else:
        print(f"  ✅ Using target column: {target_col}")

    # --- Clean and filter ---
    df = df.replace([np.inf, -np.inf], np.nan)

    # Keep numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df[numeric_cols].copy()
    df_numeric = df_numeric.dropna(axis=1, how='all')

    if target_col not in df_numeric.columns and target_col in df.columns:
        df_numeric[target_col] = df[target_col]

    imputer = SimpleImputer(strategy='mean')
    df_numeric[df_numeric.columns] = imputer.fit_transform(df_numeric)

    y = df_numeric[target_col]
    X = df_numeric.drop(columns=[target_col], errors="ignore")

    if X.empty or y.empty:
        print(f"  ⚠️ Skipping {ticker} — insufficient valid samples.")
        continue

    # --- Split and Train ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
        metrics = evaluate_classification(y_test, preds, y_proba)
        metrics.update({"Ticker": ticker, "Model": name})
        results.append(metrics)
    
        print(f"  → {name}: Acc={metrics['Accuracy']:.3f}, F1={metrics['F1']:.3f}, Prec={metrics['Precision']:.3f}, Rec={metrics['Recall']:.3f}")
    
        # --- Save trained model ---
        model_filename = models_dir / f"{ticker}_{name}_classification.pkl"
        joblib.dump(model, model_filename)
        print(f"  💾 Saved model: {model_filename}")

# -------------------------------
# Save Results
# -------------------------------
if results:
    results_df = pd.DataFrame(results)
    save_path = results_dir / "final_classification_results.csv"
    results_df.to_csv(save_path, index=False)
    print("\n✅ Classification completed. Results saved to:", save_path)
    display(results_df.sort_values(["Ticker", "F1"], ascending=[True, False]))
else:
    print("\n⚠️ No valid results to display.")



=== Processing RELIANCE ===
  Loaded dataset shape: (1460, 54)
  ✅ Using target column: Target_Cls
  → LogisticRegression: Acc=0.726, F1=0.726, Prec=0.774, Rec=0.684
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_LogisticRegression_classification.pkl
  → DecisionTree: Acc=0.469, F1=0.582, Prec=0.500, Rec=0.697
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_DecisionTree_classification.pkl
  → RandomForest: Acc=0.497, F1=0.524, Prec=0.526, Rec=0.523
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_RandomForest_classification.pkl
  → SVM: Acc=0.531, F1=0.694, Prec=0.531, Rec=1.000
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_SVM_classification.pkl

=== Processing TCS ===
  Loaded dataset shape: (1460, 54)
  ✅ Using target column: Target_Cls
  → LogisticRegression: Acc=0.702, F1=0.707, Prec=0.686, Rec=0.729
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\TCS_LogisticRegression_classificatio

,Accuracy,Precision,Recall,F1,ROC_AUC,Ticker,Model
8,0.732877,0.778571,0.698718,0.736486,0.843137,HDFCBANK,LogisticRegression
11,0.534247,0.534247,1.000000,0.696429,0.468844,HDFCBANK,SVM
10,0.565068,0.616000,0.493590,0.548043,0.581283,HDFCBANK,RandomForest
9,0.541096,0.579710,0.512821,0.544218,0.543175,HDFCBANK,DecisionTree
0,0.726027,0.773723,0.683871,0.726027,0.814175,RELIANCE,LogisticRegression
3,0.530822,0.530822,1.000000,0.693512,0.437909,RELIANCE,SVM
1,0.469178,0.500000,0.696774,0.582210,0.454227,RELIANCE,DecisionTree
2,0.496575,0.525974,0.522581,0.524272,0.508147,RELIANCE,RandomForest
4,0.702055,0.686275,0.729167,0.707071,0.783502,TCS,LogisticRegression
7,0.493151,0.493151,1.000000,0.660550,0.432432,TCS,SVM
